# Laboratorio 6. Regresión logística

En este notebook construiremos un modelo de **regresión logística** para un problema de clasificación binaria utilizando un dataset incluido en `scikit-learn`.

Trabajaremos con el dataset **Breast Cancer Wisconsin**, cuyo objetivo es clasificar tumores como benignos o malignos a partir de características obtenidas de imágenes de masas celulares.

## Carga de datos

Primero cargamos el dataset utilizando `load_breast_cancer`.

El objeto contiene tanto los predictores como la variable respuesta y los nombres de las variables.


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

data = load_breast_cancer()

In [ ]:
print(data.DESCR)

Construimos la matriz de predictores `X` y el vector de respuesta `y`.

In [ ]:
data.target_names

Construimos la matriz de predictores `X` y el vector de respuesta `y`.

In [ ]:
X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

## Exploración de datos

Antes de ajustar el modelo, revisamos la estructura de los predictores y la respuesta

In [ ]:
X.head()

In [ ]:
X.shape

In [ ]:
y.value_counts()

## Separación de conjuntos de entrenamiento y prueba

Dividiremos los datos en un conjunto de entrenamiento y un conjunto de prueba.

Utilizamos `stratify=y` para conservar aproximadamente la misma proporción de clases en ambos conjuntos.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

## Ajustar la regresión logística

Ahora ajustamos el modelo utilizando los datos de entrenamiento.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=10000
)

model.fit(X_train, y_train)

El modelo estima un intercepto $\beta_0$ y un coeficiente $\beta_i$ para cada predictor.

Podemos observar los coeficientes:

In [ ]:
coeficientes = pd.DataFrame({
    "Variable": X.columns,
    "Beta": model.coef_[0]
})

coeficientes

## Interpretar los coeficientes

En regresión logística:

$$
\log\left(\frac{p}{1-p}\right)
=
\beta_0+\beta_1x_1+\cdots+\beta_px_p
$$

Un incremento de una unidad en $x_i$, manteniendo los demás predictores constantes, cambia los **log-odds** en $\beta_i$.

Al aplicar la función exponencial:

$$
e^{\beta_i}
$$

obtenemos el factor por el cual se multiplican los **odds**.

In [ ]:
coeficientes["Odds Ratio"] = np.exp(
    coeficientes["Beta"]
)

coeficientes

## Probabilidades estimadas

La regresión logística no produce directamente una clase. El modelo estima la probabilidad de que cada observación pertenezca a la **clase positiva**.

En nuestro caso:

$$
P(Y=1\mid X)
=
P(\text{benign}\mid X)
$$

Podemos obtener estas probabilidades utilizando `predict_proba()`:


In [ ]:
probabilidades = model.predict_proba(X_test)

probabilidades[:5]

El resultado contiene una columna para cada clase:

$$
P(Y=0\mid X)
\qquad
P(Y=1\mid X)
$$

Podemos verificar el orden de las clases utilizado por el modelo:

In [ ]:
model.classes_

Como la clase `1` corresponde a **benign**, la segunda columna contiene:
$$
P(Y=1\mid X)
=
P(\text{benign}\mid X)
$$

In [ ]:
p_benign = probabilidades[:, 1]

p_benign[:10]

## Del modelo a la clasificación

Para convertir las probabilidades estimadas en clases debemos establecer un **umbral de decisión** $c$.

Utilizando la clase `1` como referencia:

$$
\hat{Y} =
\begin{cases}
1, & \hat{p} \geq c,\\
0, & \hat{p} < c.
\end{cases}
$$

Comenzaremos con:

$$
c=0.5
$$

In [ ]:
c = 0.5

y_pred = (p_benign >= c).astype(int)

y_pred[:10]

Por lo tanto:

- si $\hat p \geq 0.5$, clasificamos la observación como **benigna**;
- si $\hat p < 0.5$, la clasificamos como **maligna**.

Podemos comparar las probabilidades, predicciones y clases reales:

In [ ]:
resultados = pd.DataFrame({
    "Probabilidad": p_benign,
    "Predicción": y_pred,
    "Real": y_test.values
})

resultados.head(10)

## Matriz de confusión

Ahora comparamos las clases predichas con las clases reales.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm

`scikit-learn` utiliza por defecto la clase `1` como positiva, por lo que la matriz se organiza como:

```text
[[TN, FP],
 [FN, TP]]
```

Podemos extraer sus componentes:

In [ ]:
TN, FP, FN, TP = cm.ravel()

print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print("TP:", TP)

## Métricas de clasificación

A partir de la matriz de confusión podemos calcular diferentes métricas.

### Accuracy

El **accuracy** representa la proporción total de observaciones clasificadas correctamente:

$$
\text{Accuracy}
=
\frac{TP+TN}{TP+TN+FP+FN}
$$

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

accuracy

### Precision

La **precision** responde:

> De todos los tumores clasificados como benignos, ¿qué proporción realmente era benigna?

$$
\text{Precision}
=
\frac{TP}{TP+FP}
$$

In [ ]:
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred)

precision

### Recall

El **recall** responde:

> De todos los tumores realmente benignos, ¿qué proporción fue identificada correctamente como benigna?

$$
\text{Recall}
=
\frac{TP}{TP+FN}
$$

In [ ]:
from sklearn.metrics import recall_score

recall = recall_score(y_test, y_pred)

recall

In [ ]:
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

## Curva ROC

Hasta ahora evaluamos el clasificador utilizando un único umbral:

$$
c = 0.5
$$

Sin embargo, diferentes valores de $c$ producen diferentes cantidades de verdaderos positivos y falsos positivos.

La **curva ROC** permite estudiar el comportamiento del clasificador para distintos valores del umbral.

Para cada umbral calculamos:

$$
\operatorname{TPR}
=
\frac{TP}{TP+FN}
$$

y

$$
\operatorname{FPR}
=
\frac{FP}{FP+TN}
$$

donde:

- `TPR` corresponde al **recall** de la clase positiva;
- `FPR` representa la proporción de observaciones negativas clasificadas incorrectamente como positivas.

En este dataset, la clase positiva es:

$$
1 = \text{benign}
$$

Podemos calcular estos valores utilizando `roc_curve()`:

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(
    y_test,
    p_benign
)

In [ ]:
roc = pd.DataFrame({
    "Threshold": thresholds,
    "FPR": fpr,
    "TPR": tpr
})

roc.head(10)

Cada fila representa un umbral diferente y genera un punto:

$$
(\operatorname{FPR},\operatorname{TPR})
$$

La curva ROC se obtiene al representar todos estos puntos.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], "--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.show()

## Área bajo la curva

La curva ROC puede resumirse utilizando el **Area Under the Curve (AUC)**.

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    p_benign
)

auc

El AUC toma valores entre `0` y `1`.

En general:

- un valor cercano a `1` indica una alta capacidad de discriminación;
- un valor cercano a `0.5` indica un desempeño similar al de una clasificación aleatoria.

El AUC evalúa la capacidad del modelo para separar las clases considerando distintos valores del umbral.